In [1]:
from pathlib import Path
import random
import shutil
import xml.etree.ElementTree as ET

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Reproducibility
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# Project paths
PROJECT_ROOT = Path.cwd().parent

ORIGINAL_DIR = PROJECT_ROOT / "data" / "dataset"
OUTPUT_DIR = PROJECT_ROOT / "data" / "yolo_dataset"

TRAIN_DIR = ORIGINAL_DIR / "train"
TEST_DIR = ORIGINAL_DIR / "test"

print("Project root:", PROJECT_ROOT)
print("Original dataset:", ORIGINAL_DIR)
print("Output dataset:", OUTPUT_DIR)

print("Original dataset exists:", ORIGINAL_DIR.exists())
print("Training folder exists:", TRAIN_DIR.exists())
print("Test folder exists:", TEST_DIR.exists())

Project root: c:\Users\ribkh\plate-recognition
Original dataset: c:\Users\ribkh\plate-recognition\data\dataset
Output dataset: c:\Users\ribkh\plate-recognition\data\yolo_dataset
Original dataset exists: True
Training folder exists: True
Test folder exists: True


In [2]:
train_images = sorted(TRAIN_DIR.glob("*.jpg"))
test_images = sorted(TEST_DIR.glob("*.jpg"))

train_xml = sorted(TRAIN_DIR.glob("*.xml"))
test_xml = sorted(TEST_DIR.glob("*.xml"))

print("Training images:", len(train_images))
print("Training annotations:", len(train_xml))

print("Test images:", len(test_images))
print("Test annotations:", len(test_xml))

Training images: 567
Training annotations: 567
Test images: 142
Test annotations: 142


In [3]:
def check_matching_files(folder):

    images = {file.stem for file in folder.glob("*.jpg")}
    annotations = {file.stem for file in folder.glob("*.xml")}

    return {
        "missing_annotations": sorted(images - annotations),
        "missing_images": sorted(annotations - images)
    }


print("TRAIN:", check_matching_files(TRAIN_DIR))
print("TEST:", check_matching_files(TEST_DIR))

TRAIN: {'missing_annotations': [], 'missing_images': []}
TEST: {'missing_annotations': [], 'missing_images': []}


In [4]:
# Create YOLO dataset directories

folders = [
    OUTPUT_DIR / "images" / "train",
    OUTPUT_DIR / "images" / "val",
    OUTPUT_DIR / "images" / "test",

    OUTPUT_DIR / "labels" / "train",
    OUTPUT_DIR / "labels" / "val",
    OUTPUT_DIR / "labels" / "test"
]

for folder in folders:
    folder.mkdir(parents=True, exist_ok=True)

print("YOLO dataset directories created successfully!")

YOLO dataset directories created successfully!


In [5]:
from sklearn.model_selection import train_test_split

# Get all original training image paths
all_train_images = sorted(TRAIN_DIR.glob("*.jpg"))

# Split: 80% training, 20% validation
train_files, val_files = train_test_split(
    all_train_images,
    test_size=0.20,
    random_state=RANDOM_SEED,
    shuffle=True
)

print("Training images:", len(train_files))
print("Validation images:", len(val_files))

ModuleNotFoundError: No module named 'sklearn'

In [ ]:
def copy_image_annotation_pairs(files, destination):

    for image_path in files:

        xml_path = image_path.with_suffix(".xml")

        shutil.copy2(
            image_path,
            destination / image_path.name
        )

        shutil.copy2(
            xml_path,
            destination / xml_path.name
        )


# Copy training and validation pairs
copy_image_annotation_pairs(
    train_files,
    OUTPUT_DIR / "images" / "train"
)

copy_image_annotation_pairs(
    val_files,
    OUTPUT_DIR / "images" / "val"
)

# Copy original test pairs without splitting them
test_files = sorted(TEST_DIR.glob("*.jpg"))

copy_image_annotation_pairs(
    test_files,
    OUTPUT_DIR / "images" / "test"
)

print("Files copied successfully!")

In [ ]:
for split in ["train", "val", "test"]:

    folder = OUTPUT_DIR / "images" / split

    images = list(folder.glob("*.jpg"))
    annotations = list(folder.glob("*.xml"))

    print(
        f"{split.upper()}: "
        f"{len(images)} images | "
        f"{len(annotations)} XML annotations"
    )